# Email Support 7B Model — Fine-tuning on Google Colab

**Target Model:** `Qwen/Qwen2.5-7B-Instruct` (Fine-tuned for ArtCloud KB)
**Method:** QLoRA 4-bit via **Unsloth** — trains on free T4 GPU in ~20-40 minutes
**Output:** GGUF Q4_K_M — loads directly into Ollama as `email-support-7b`

### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `7b_kb_train.jsonl` from your machine (generated by `prepare_7b_data.py`)


In [ ]:
# Verify GPU is available
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode == 0:
    print('GPU:', r.stdout.strip())
else:
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4 GPU')

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
print('Done.')

## Step 2 — Upload `7b_kb_train.jsonl`

In [ ]:
from google.colab import files
print('Upload 7b_kb_train.jsonl from your data/training folder:')
uploaded = files.upload()
assert '7b_kb_train.jsonl' in uploaded, 'ERROR: File must be named 7b_kb_train.jsonl'
print(f'Uploaded {len(uploaded["7b_kb_train.jsonl"])/1024:.1f} KB')

## Step 3 — Load Qwen 2.5 7B Model

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

print(f'Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Step 4 — Apply LoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
model.print_trainable_parameters()

## Step 5 — Prepare ArtCloud Dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset('json', data_files='7b_kb_train.jsonl', split='train')

def format_record(row):
    messages = [
        {"role": "system",    "content": row['instruction']},
        {"role": "user",      "content": row['input']},
        {"role": "assistant", "content": row['output']},
    ]
    # Unsloth handles the Qwen2.5 template automatically
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = raw.map(format_record, remove_columns=raw.column_names)

print('\nSample formatted record:')
print(dataset[0]['text'][:500])
print('...')

## Step 6 — Train (Knowledge Injection)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = 'text',
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_num_proc            = 2,
        packing                     = True,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 10,   # More epochs for small dataset to ensure memorization
        warmup_steps                = 5,
        learning_rate               = 1e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 1,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'linear',
        seed                        = 42,
        output_dir                  = '/content/checkpoints',
        report_to                   = 'none',
    ),
)

print('Starting training...')
stats = trainer.train()
print(f'\nFinal loss: {stats.training_loss:.4f}')

## Step 7 — Save DIRECTLY to Google Drive (Safe from Crashes)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print('Saving GGUF model to your Google Drive (Q4_K_M)...')
print('This will take about 15-20 minutes. KEEP THIS TAB OPEN and stay active!')

model.save_pretrained_gguf(
    '/content/drive/MyDrive/email-support-7b-gguf',
    tokenizer,
    quantization_method = 'q4_k_m'
)

import os
output_path = '/content/drive/MyDrive/email-support-7b-gguf'
if os.path.exists(output_path):
    print(f'\nSUCCESS! Your model is saved in Drive: {os.listdir(output_path)}')
    print('\nNext step: Go to drive.google.com and download the .gguf file to your computer.')
else:
    print('\nERROR: Model not found. Check Drive space.')